# ChronoQuant — LightGBM Hyperparameter Search (Colab)

Ez a notebook futtatja a LightGBM keresési fázist (smoke + explore) a long és short modellekre.

**Futtatás előtt helyi gépen (Claude végzi):**
1. Új sample létrehozása: `python scripts/create_sample_splits.py ...`
2. Új model ID-k hozzáadása `config/models.json`-hoz (`active: false`)
3. Parquet export: `python scripts/export_sample_parquet.py --sample-id <id> --copy-to-drive`
4. Commit + push a GitHub-ra

**Futtatás:**
1. Frissítsd a **CONFIG** cellát (model ID-k, sample ID)
2. Futtasd az összes cellát: **Ctrl+F9** (Run All)
3. Keresés végén az artifacts automatikusan visszakerülnek Drive-ra

**Utána helyi gépen (Claude végzi):**
- Artifacts másolása Drive-ról → `models/<model_id>/search/`
- Search review + feature importance
- Final fit, prediction sync, strategy sweep, promotion


In [ ]:
# =============================================================================
# CONFIG — frissítsd minden új fejlesztési ciklus előtt
# =============================================================================

LONG_MODEL_ID  = "lgbm_solusdt_l_fw60_q90_local_v4"
SHORT_MODEL_ID = "lgbm_solusdt_s_fw60_q10_local_v4"
SAMPLE_ID      = "base_solusdt_fw60_futures_v1"
ASSET_ID       = "solusdt_fw60"

REPO_URL       = "https://github.com/Nemesis14/chronoquant.git"
REPO_BRANCH    = "main"
REPO_PATH      = "/content/chronoquant"
DRIVE_ROOT     = "/content/drive/My Drive/chronoquant"
DB_PATH        = f"{REPO_PATH}/database/solusdt_data_dev.db"

# Keresési paraméterek
EXPLORE_TRIALS    = 60
EXPLORE_TIMEOUT_H = 3.0

print(f"Long model:  {LONG_MODEL_ID}")
print(f"Short model: {SHORT_MODEL_ID}")
print(f"Sample:      {SAMPLE_ID}")

In [ ]:
# =============================================================================
# 1. Drive csatolás
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')
print("OK: Drive csatolva")

In [ ]:
# =============================================================================
# 2. Repo klónozása / frissítése GitHub-ról
# =============================================================================
import os
from pathlib import Path

if Path(f"{REPO_PATH}/.git").exists():
    print("INFO: Repo megvan, frissítés...")
    os.system(f"git -C {REPO_PATH} fetch origin")
    os.system(f"git -C {REPO_PATH} reset --hard origin/{REPO_BRANCH}")
else:
    print("INFO: Repo klónozása...")
    os.system(f"git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_PATH}")

commit = os.popen(f"git -C {REPO_PATH} log --oneline -1").read().strip()
print(f"OK: Repo kész  [{commit}]")

In [ ]:
# =============================================================================
# 3. Függőségek telepítése
# =============================================================================
!pip install -q "lightgbm>=4.6.0" pandas numpy "scikit-learn>=1.8.0" pyarrow ta statsmodels
print("OK: Csomagok telepítve")

In [ ]:
# =============================================================================
# 4. Parquet → SQLite (chunked, alacsony RAM) + config patchelése
# =============================================================================
import json
import sqlite3
import sys
import time
from pathlib import Path

import pyarrow.parquet as pq

sys.path.insert(0, f"{REPO_PATH}/src")

# --------- Parquet megnyitása Drive-ról (PyArrow batch reader, nincs /tmp copy) ---------
drive_parquet = f"{DRIVE_ROOT}/samples/{SAMPLE_ID}/dataset.parquet"
print(f"INFO: Parquet megnyitása: {drive_parquet}")
parquet_file  = pq.ParquetFile(drive_parquet)
total_rows    = parquet_file.metadata.num_rows
print(f"INFO: {total_rows:,} sor, {parquet_file.metadata.num_columns} oszlop")

# --------- SQLite létrehozása chunkonként (100k sor/chunk, ~300 MB RAM) ---------
BATCH_SIZE = 100_000
Path(DB_PATH).parent.mkdir(parents=True, exist_ok=True)

written = 0
t0 = time.time()
with sqlite3.connect(DB_PATH) as conn:
    for i, batch in enumerate(parquet_file.iter_batches(batch_size=BATCH_SIZE)):
        chunk = batch.to_pandas()
        chunk.to_sql(
            "solusdt_1m_features",
            conn,
            if_exists = "replace" if i == 0 else "append",
            index     = False,
        )
        written += len(chunk)
        print(f"  INFO: {written:,} / {total_rows:,} sor írva...", end="\r")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_open_time ON solusdt_1m_features(open_time)")

elapsed = time.time() - t0
print(f"\nOK: SQLite kész — {written:,} sor, {elapsed:.0f}s  [{DB_PATH}]")

# --------- config/assets.json patchelése ---------
assets_path = Path(f"{REPO_PATH}/config/assets.json")
assets_cfg  = json.loads(assets_path.read_text())
assets_cfg["assets"][ASSET_ID]["db_path"] = DB_PATH
assets_path.write_text(json.dumps(assets_cfg, indent=4))

# --------- samples/<id>/metadata.json patchelése ---------
meta_path = Path(f"{REPO_PATH}/samples/{SAMPLE_ID}/metadata.json")
meta      = json.loads(meta_path.read_text())
meta["source"]["db_path"] = DB_PATH
meta_path.write_text(json.dumps(meta, indent=4))

print("OK: Config patchelve")

In [ ]:
# =============================================================================
# 5. Feature tábla audit
# =============================================================================
import sqlite3
import sys
import pandas as pd

sys.path.insert(0, f"{REPO_PATH}/src")
import utils

with sqlite3.connect(DB_PATH) as conn:
    info = pd.read_sql_query(
        "SELECT "
        "  MIN(open_time) as data_start, "
        "  MAX(open_time) as data_end, "
        "  COUNT(*) as total_rows, "
        "  SUM(CASE WHEN trg_l_fw60_q90 IS NOT NULL THEN 1 ELSE 0 END) as labeled_l, "
        "  SUM(CASE WHEN trg_s_fw60_q10 IS NOT NULL THEN 1 ELSE 0 END) as labeled_s "
        "FROM solusdt_1m_features",
        conn,
    )
print(info.to_string(index=False))

# Modell registry ellenőrzése
models_cfg = utils.load_models_config()
models     = models_cfg.get("models", {})
for mid in [LONG_MODEL_ID, SHORT_MODEL_ID]:
    status = "OK" if mid in models else "ERROR — nincs a config/models.json-ban!"
    print(f"{status}: {mid}")

missing = [mid for mid in [LONG_MODEL_ID, SHORT_MODEL_ID] if mid not in models]
if missing:
    raise RuntimeError(f"Model ID-k hiányoznak a registryből: {missing}")

In [ ]:
# =============================================================================
# 6a. Smoke test — LONG modell
# =============================================================================
import os
os.chdir(REPO_PATH)

print(f"=== SMOKE: {LONG_MODEL_ID} ===")
!python scripts/search_lgbm.py --model-id {LONG_MODEL_ID} --stage smoke

In [ ]:
# =============================================================================
# 6b. Explore — LONG modell  (60 trial, ~2-3 ora)
# =============================================================================
print(f"=== EXPLORE: {LONG_MODEL_ID} ({EXPLORE_TRIALS} trial, {EXPLORE_TIMEOUT_H}h max) ===")
!python scripts/search_lgbm.py \
    --model-id {LONG_MODEL_ID} \
    --stage explore \
    --n-trials {EXPLORE_TRIALS} \
    --timeout-hours {EXPLORE_TIMEOUT_H}

In [ ]:
# =============================================================================
# 7a. Smoke test — SHORT modell
# =============================================================================
print(f"=== SMOKE: {SHORT_MODEL_ID} ===")
!python scripts/search_lgbm.py --model-id {SHORT_MODEL_ID} --stage smoke

In [ ]:
# =============================================================================
# 7b. Explore — SHORT modell  (60 trial, ~2-3 ora)
# =============================================================================
print(f"=== EXPLORE: {SHORT_MODEL_ID} ({EXPLORE_TRIALS} trial, {EXPLORE_TIMEOUT_H}h max) ===")
!python scripts/search_lgbm.py \
    --model-id {SHORT_MODEL_ID} \
    --stage explore \
    --n-trials {EXPLORE_TRIALS} \
    --timeout-hours {EXPLORE_TIMEOUT_H}

In [ ]:
# =============================================================================
# 8. Artifacts mentése Drive-ra
# =============================================================================
import shutil

for model_id in [LONG_MODEL_ID, SHORT_MODEL_ID]:
    src = Path(f"{REPO_PATH}/models/{model_id}")
    dst = Path(f"{DRIVE_ROOT}/models/{model_id}")
    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        files = list(dst.rglob("*"))
        print(f"OK: {model_id} -> {dst}  ({len(files)} fajl)")
    else:
        print(f"WARN: {model_id} — artifacts nem talalhatok, ellenorizd a keresesi logot")

print()
print("=== KESZ ===")
print("Kovetkezo lepesek helyi gepen: lasd a notebook tetején levo instrukciokat.")

## Következő lépések (helyi gépen, Claude végzi)

### Artifacts másolása
```bash
# Claude átmásolja Drive-ról:
# F:\My Drive\chronoquant\models\<model_id>\  →  models\<model_id>\
```

### Search review
```bash
python -c "
import json
for mid in ['lgbm_solusdt_l_fw60_q90_local_v4', 'lgbm_solusdt_s_fw60_q10_local_v4']:
    best = json.load(open(f'models/{mid}/search/search_best.json'))
    print(mid, '  prauc:', round(best.get('mean_valid_prauc',0),4), '  ll:', round(best.get('mean_valid_ll',0),4))
"
```

### Final fit + prediction sync + strategy sweep
Lásd: `docs/engineering/lgbm_model_development.md` — 5–8. lépések

### Promotion
```bash
# config/models.json  → active: true az új, false a régi
# config/env.json     → runtime model_id frissítés
# config/strategies.json → új strategy bejegyzés
```
